# Deep Q-learning

Neste notebook vamos ver o funcionamento do algoritmo Deep Q-learning usando a biblioteca `stable_baselines3`, que nos fornece vários algoritmos de _Deep Learning_ e _Reinforcement Learning_.

Além disso, usaremos a biblioteca `gymnasium`, da OpenAI, que nos fornece cenários e ambientes interessantes para testarmos esse algoritmo.

## Setup

### Imports

In [ ]:
import os
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.logger import configure
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display

### Usar GPU para o Treinamento (MPS)

[Mac Apple Silicon only] Apple's _Metal Performance Shaders_ backend

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

print("Selected device:", device)

## Configurar Modelo

Nesse primeiro exercício vamos explorar o cenário do [CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/). Conforme visto em aula, o Deep Q-Learning usa uma rede neural profunda na sua arquitetura para aprender.

Portanto, definiremos `POLICY = "MlpPolicy"` para o uso de uma rede do tipo _Multi-layer Perceptron_.

In [ ]:
POLICY = "MlpPolicy"

Inicializamos o ambiente `env` com `render_mode = "rgb_array"`, possibilitando visualizar um vídeo dos resultados após o treino.

In [ ]:
env = gym.make(
    id="CartPole-v1",  # env name
    render_mode="rgb_array"
)

Aqui definimos uma função para inicializar o modelo de uma Deep Q-network com os parâmetros padrões.

In [ ]:
def create_model(env) -> DQN:
    """
    Initialize a DQN model with core hyperparameters.

    :return: A pre-configured instance of a Deep Q-learning model.
    :rtype: DQN
    """

    return DQN(
        policy=POLICY,
        env=env,
        device=device,                   # Usar GPU (Apple Silicon MPS)
        learning_rate=5e-4,              # Tamanho do "step" do otimizador (gradiente)
        gamma=0.99,                      # Fator de desconto (recompensas futuras)
        buffer_size=10000,               # Tamanho do buffer de replay
        learning_starts=1000,            # Delay antes de mandar atualizações
        batch_size=64,                   # Tamanho do batch do otimizador (gradiente)
        train_freq=4,                    # Treina a cada n passos
        target_update_interval=1000,     # Frequência para atualizar a rede alvo
        exploration_fraction=0.01,       # Determina o declínio do epsilon
        exploration_initial_eps=1.0,     # Começa com exploração no máximo
        exploration_final_eps=0.02,      # Exploração mínima no final
        verbose=1,
        tensorboard_log="./logs/"
    )

Abaixo segue o script para treinar a rede.

Ao final, salvamos o modelo treinado num arquivo `zip` para ser usado novamente no futuro.

In [ ]:
def train_agent(model, total_timesteps: int=200000):
    """Train the DQN agent."""

    print("Starting training...\n")

    model.learn(total_timesteps=total_timesteps, log_interval=10)
    model.save("models/dqn_cartpole_model")

    print("\nTraining complete. Model saved at 'models/dqn_cartpole_model.zip'")

Definimos aqui uma função que testa o modelo treinado e gera um vídeo para que possamos observar o agente funcionando no ambiente.

In [ ]:
def run_agent(env, n_episodes: int=1):
    """Run the trained agent for multiple demo episodes."""

    print(f"Starting demo mode for {n_episodes} episodes...\n")

    videos_dir = "videos"
    env = RecordVideo(
        env,
        video_folder=videos_dir,
        episode_trigger=lambda e: True
    )
    model = DQN.load("models/dqn_cartpole_model", env=env, device=device)
    total_rewards = []

    for episode in range(n_episodes):
        obs, _ = env.reset()
        done = False
        episode_reward = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward  # type: ignore (Pylance)

        total_rewards.append(episode_reward)
        print(f"Episode {episode + 1} | Reward: {episode_reward:.2f}")

    mean_reward = sum(total_rewards) / len(total_rewards)
    env.close()
    print(f"\n Demo finished | Mean reward over {n_episodes} episodes: {mean_reward:.2f}")

    video_files = [
        f for f in os.listdir(videos_dir)
        if f.endswith(".mp4")
    ]

    for video_file in video_files:
        display(Video(
            os.path.join(videos_dir, video_file),
            embed=True
        ))

## Execução

### Treinar o Modelo

In [ ]:
logger = configure(
    folder="./logs/",
    format_strings=["stdout", "tensorboard"]
)
model = create_model(env)
train_agent(model, total_timesteps=200000)

### Rodar o Modelo

In [ ]:
run_agent(env)

## Exercício

### 1 - O agente aprende uma estratégia válida? Justifique.

Sim, o agente aprendeu uma estratégia válida.

O ambiente `CartPole-v1` _trunca_ o episódio após 500 passos, portanto a recompensa máxima possível por episódio é 500.

Isso significa que o agente conseguiu manter a haste equilibrada durante todo o episódio sem derrubá-la, que é exatamente o objetivo da tarefa.

---

### 2 - Aproximadamente quantos passos de treinamento são necessários para que o agente comece a aprender?

O aprendizado significativo começa por volta de 6.000 a 10.000 passos, conforme evidenciado pela progressão das recompensas médias nos logs de treinamento:

| Passos | Recompensa Média |
|----------:|----------------:|
| ~1.000    | ~20             |
| ~6.100    | 28,9            |
| ~7.300    | 39,1            |
| ~8.800    | 52,7            |
| ~10.800   | 70,1            |
| ~15.300   | 111             |

Os primeiros ~5.000 passos funcionam como uma fase de "aquecimento". Ou seja, o agente coleta experiências, o _epsilon_ cai rapidamente, e os gradientes iniciais têm baixo sinal. A curva de aprendizado decola de forma clara entre 6.000 e 10.000 passos.

---

### 3 - O comportamento aprendido é estável ou errático? Justifique.

O comportamento aprendido é **errático durante o treinamento**, mas converge para uma política estável no final.

Apesar da instabilidade no treinamento, a execução final -- `run_agent()` -- atingiu recompensa 500/500, indicando que o modelo salvo capturou um ponto estável da política antes das oscilações finais.

Em suma: o treinamento é errático, mas a política aprendida no momento do _save_ do modelo é sólida.

---

### 4 - Se você executar o treinamento da rede múltiplas vezes, obtém sempre o mesmo resultado? Justifique.

Não, executar o treinamento múltiplas vezes não produz garante o mesmo resultado. O aprendizado por reforço profundo é intrinsecamente estocástico, e este notebook não fixa nenhuma semente aleatória -- _random seed_ -- o que aumenta a variabilidade.

Na prática, em _Reinforcement Learning_ aceita-se a variabilidade e avalia-se o desempenho médio sobre múltiplas execuções independentes.